# 03 — Evaluation (M8 freeze, M9 decomposition)

CPU only; runs locally on the laptop after Kaggle results are merged (`python -m groundedgen.kaggle merge ghd_outputs.zip configs/local.yaml`).

1. **Freeze (M8)** once, after dev thresholds exist and prompts are final on dev. Only then run test generation on Kaggle.
2. **Evaluate (M9)**: per-pair outcomes, origins (EP/GD/GF), leakage, inflation, paired tests, T3/T4/T5/T7 and F3/F4.

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import groundedgen  # noqa: F401
from groundedgen import utils
CONFIGS = ["configs/local.yaml"] if os.path.exists("configs/local.yaml") else []
cfg = utils.load_config(extra_paths=CONFIGS)
print(cfg["project_dir"])

## M8 — freeze prompts and thresholds

Run once. It refuses if any extractor lacks dev thresholds or if `frozen.json` already exists.

In [ ]:
from groundedgen import generate
FREEZE = False   # set True deliberately, once
if FREEZE:
    print(json.dumps(generate.freeze(cfg), indent=2))
else:
    print("frozen.json exists:", generate.frozen_path(cfg["project_dir"]).exists())

## M9 — evaluation

`SPLIT = "dev"` works any time for checking; `"test"` needs `frozen.json`.

In [ ]:
from groundedgen import analysis, claims
SPLIT = "dev"
for gen_file in sorted(utils.project_path(cfg, "outputs", "generation").rglob("*.jsonl")):
    claims.run_claims(cfg["project_dir"], gen_file)
print(analysis.run_evaluation(cfg, SPLIT))

In [ ]:
import pandas as pd
from IPython.display import Image, display
adir = utils.project_path(cfg, "outputs", "analysis")
sfx = "" if SPLIT == "test" else f"_{SPLIT}"
pd.set_option("display.max_rows", 200)
for name in ("T3_main", "T4_origins", "T5_abstention", "T7_ethnic"):
    print(name); display(pd.read_csv(adir / f"{name}{sfx}.csv"))
display(pd.read_csv(adir / f"tests_mcnemar{sfx}.csv"))
for fig in ("F3_origins", "F4_tradeoff"):
    display(Image(filename=str(adir / f"{fig}.png")))